
# k-NN
Cover & Hart (1967): https://ieeexplore.ieee.org/document/1053964




FAISS (2021): https://arxiv.org/abs/1702.08734





Metric Learning Reality Check (2020): https://arxiv.org/abs/2003.08505

# k-Nearest Neighbours

In [1]:
# Arabic Sentiment Analysis — k-Nearest Neighbours (Cover & Hart, 1967)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

# For TF-IDF + k-NN, cosine similarity is the natural metric.
# sklearn KNN uses 'cosine' via metric parameter.
K_NEIGHBOURS = 17

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# sklearn KNN with metric='cosine' supports sparse matrices
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])

print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. MODEL — k-NN (Cover & Hart)
# Cosine distance is preferred over Euclidean for TF-IDF vectors because
# it is invariant to document length (magnitude of the TF-IDF vector).
# algorithm='brute' is required for cosine metric on sparse data.
# weights='distance' gives closer neighbours more influence.
print("=" * 60)
print(f"  🔵 Training: k-NN  (k={K_NEIGHBOURS}, metric=cosine)")
print("=" * 60)

clf = KNeighborsClassifier(
    n_neighbors = K_NEIGHBOURS,
    metric      = "cosine",
    algorithm   = "brute",    # exact search; needed for cosine + sparse
    weights     = "distance", # inverse-distance weighting
    n_jobs      = -1,
)
clf.fit(X_train, y_train)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  k-NN (k={K_NEIGHBOURS})  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 7,726
Final feature shape (train) : (5067, 7728)

Classes : [0 1]  →  [0, 1]

  🔵 Training: k-NN  (k=17, metric=cosine)

────────────────────────────────────────────────────────────
  k-NN (k=17)  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.7919
  Precision : 0.7940
  Recall    : 0.7919
  F1-Score  : 0.7915

  Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.75      0.78       543
           1       0.77      0.83      0.80       543

    accuracy                           0.79      1086
   macro avg       0.79      0.79      0.79      1086
weighted avg       0.79      0.79      0.79      1086

  Confusion Matrix:
[[407 136]
 [ 90 453]]


───────────────────────────────────────────────────────

# FAISS k-NN

In [2]:
# Arabic Sentiment Analysis — FAISS k-NN (Approximate Nearest Neighbours)


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False
    print("⚠️  FAISS not found. Install with:  pip install faiss-cpu")
    print("   Falling back to exact cosine k-NN via sklearn.\n")
    from sklearn.neighbors import KNeighborsClassifier

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

K_NEIGHBOURS  = 17   # number of nearest neighbours to retrieve
N_LISTS       = 100  # IVF: number of Voronoi cells (≈ sqrt(n_train))
N_PROBE       = 10   # IVF: cells to visit at query time (speed/accuracy trade-off)

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# FAISS requires float32 dense arrays
def to_float32(X_sparse):
    X = hstack([X_sparse, emoji_sparse(train_df if X_sparse is X_train_tfidf
                                        else (val_df if X_sparse is X_val_tfidf else test_df))])
    return X

# Build dense float32 matrices and L2-normalise for cosine via inner product
def prepare(tfidf_mat, emoji_df):
    X = hstack([tfidf_mat, emoji_sparse(emoji_df)]).toarray().astype(np.float32)
    return normalize(X, norm="l2")   # unit vectors → inner product = cosine

X_train = prepare(X_train_tfidf, train_df)
X_val   = prepare(X_val_tfidf,   val_df)
X_test  = prepare(X_test_tfidf,  test_df)

print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. MODEL — FAISS k-NN
def majority_vote(neighbour_labels: np.ndarray) -> np.ndarray:
    """Majority vote across k neighbours for each query sample."""
    return np.array([
        Counter(row).most_common(1)[0][0]
        for row in neighbour_labels
    ])

if FAISS_AVAILABLE:
    print("=" * 60)
    print("  🔵 Building FAISS IVFFlat Index (cosine via inner product)")
    print("=" * 60)

    d = X_train.shape[1]  # feature dimensionality

    # IndexFlatIP = exact inner product search (cosine on normalised vectors)
    # For large datasets, switch to IndexIVFFlat for approximate search
    n_train = X_train.shape[0]

    if n_train >= 10_000:
        # Approximate search: IVF with flat (uncompressed) vectors
        quantiser = faiss.IndexFlatIP(d)
        index     = faiss.IndexIVFFlat(quantiser, d, N_LISTS, faiss.METRIC_INNER_PRODUCT)
        index.train(X_train)
        index.add(X_train)
        index.nprobe = N_PROBE
        print(f"   Index type  : IVFFlat  |  n_lists={N_LISTS}  |  nprobe={N_PROBE}")
    else:
        # Exact search for small datasets
        index = faiss.IndexFlatIP(d)
        index.add(X_train)
        print("   Index type  : FlatIP (exact search — dataset is small)")

    print(f"   Vectors indexed : {index.ntotal:,}")
    print(f"   k               : {K_NEIGHBOURS}\n")

    def faiss_predict(X_query: np.ndarray) -> np.ndarray:
        """Search FAISS index and return majority-voted predictions."""
        _, I = index.search(X_query, K_NEIGHBOURS)   # I: (n_query, k) indices
        neighbour_labels = y_train[I]                 # (n_query, k) labels
        return majority_vote(neighbour_labels)

    val_pred  = faiss_predict(X_val)
    test_pred = faiss_predict(X_test)

else:
    # Fallback: exact cosine k-NN via sklearn
    clf = KNeighborsClassifier(
        n_neighbors=K_NEIGHBOURS, metric="cosine",
        algorithm="brute", weights="distance", n_jobs=-1
    )
    clf.fit(X_train, y_train)
    val_pred  = clf.predict(X_val)
    test_pred = clf.predict(X_test)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  FAISS k-NN (k={K_NEIGHBOURS})  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  val_pred,  "Validation Set")
evaluate(y_test, test_pred, "Test Set")

⚠️  FAISS not found. Install with:  pip install faiss-cpu
   Falling back to exact cosine k-NN via sklearn.

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 7,726
Final feature shape (train) : (5067, 7728)

Classes : [0 1]  →  [0, 1]


────────────────────────────────────────────────────────────
  FAISS k-NN (k=17)  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.7928
  Precision : 0.7948
  Recall    : 0.7928
  F1-Score  : 0.7925

  Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.75      0.78       543
           1       0.77      0.83      0.80       543

    accuracy                           0.79      1086
   macro avg       0.79      0.79      0.79      1086
weighted avg       0.79      0.79      0.79      1086

  Confusion Matrix:
[[408 1

# Metric Learning k-NN

In [3]:
# Arabic Sentiment Analysis — Metric Learning k-NN

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import (
    KNeighborsClassifier,
    NeighborhoodComponentsAnalysis,
)
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD       # LSA for dim reduction
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE   = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

K_NEIGHBOURS   = 14
LSA_COMPONENTS = 256    # dimensionality after Latent Semantic Analysis
NCA_COMPONENTS = 64     # dimensionality after NCA projection

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

X_train_raw = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val_raw   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test_raw  = hstack([X_test_tfidf,  emoji_sparse(test_df)])

print(f"Raw feature shape (train) : {X_train_raw.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. METRIC LEARNING PIPELINE
# NCA cannot work on 50K-dimensional sparse data directly.
# Step A: LSA (TruncatedSVD) reduces to 256 dims (dense, captures semantics).
# Step B: StandardScaler normalises the LSA space.
# Step C: NCA learns a further linear projection optimised for k-NN accuracy.
# Step D: k-NN classifier operates in the final NCA-projected space.

print("=" * 60)
print("  🔵 Step A: LSA dimensionality reduction …")
print("=" * 60)

lsa = TruncatedSVD(n_components=LSA_COMPONENTS, random_state=RANDOM_STATE)
X_train_lsa = lsa.fit_transform(X_train_raw)   # fit only on train
X_val_lsa   = lsa.transform(X_val_raw)
X_test_lsa  = lsa.transform(X_test_raw)
print(f"   Post-LSA shape : {X_train_lsa.shape}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_lsa)
X_val_sc   = scaler.transform(X_val_lsa)
X_test_sc  = scaler.transform(X_test_lsa)

print("\n  🔵 Step B: NCA metric learning …")
print(f"   Projecting {LSA_COMPONENTS}D → {NCA_COMPONENTS}D …")

nca = NeighborhoodComponentsAnalysis(
    n_components  = NCA_COMPONENTS,
    max_iter      = 300,
    random_state  = RANDOM_STATE,
    verbose       = 1,
)
# NCA is fit only on training data — no leakage
X_train_nca = nca.fit_transform(X_train_sc, y_train)
X_val_nca   = nca.transform(X_val_sc)
X_test_nca  = nca.transform(X_test_sc)
print(f"\n   Post-NCA shape : {X_train_nca.shape}")

print(f"\n  🔵 Step C: k-NN in NCA metric space (k={K_NEIGHBOURS}) …")
clf = KNeighborsClassifier(
    n_neighbors = K_NEIGHBOURS,
    metric      = "euclidean",  # Euclidean in NCA space ≡ learned Mahalanobis
    weights     = "distance",
    n_jobs      = -1,
)
clf.fit(X_train_nca, y_train)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Metric Learning k-NN (NCA, k={K_NEIGHBOURS})  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val_nca),  "Validation Set")
evaluate(y_test, clf.predict(X_test_nca), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 7,726
Raw feature shape (train) : (5067, 7728)

Classes : [0 1]  →  [0, 1]

  🔵 Step A: LSA dimensionality reduction …
   Post-LSA shape : (5067, 256)

  🔵 Step B: NCA metric learning …
   Projecting 256D → 64D …
Finding principal components... done in  0.96s
[NeighborhoodComponentsAnalysis]
[NeighborhoodComponentsAnalysis]  Iteration      Objective Value    Time(s)
[NeighborhoodComponentsAnalysis] ------------------------------------------
[NeighborhoodComponentsAnalysis]          1         3.382154e+03       3.27
[NeighborhoodComponentsAnalysis]          1         3.775262e+03       2.47
[NeighborhoodComponentsAnalysis]          2         3.985616e+03       2.41
[NeighborhoodComponentsAnalysis]          3         4.120950e+03       1.93
[NeighborhoodComponentsAnalysis]          4         4.223521e+03 